In [1]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [2]:
region_list = ["ARK-NZK","Vallei en Veluwe",
               "Achterhoek", "Brabantse Delta","Friesland",
               "Groningen en NO-Drenthe","Limburg",
               "Noord-Brabant Oost","Noord-Westelijke Delta",
               "Rivierenland","Scheldestromen","Zuiderzeeland",
               "Overijsselse Vecht"
               ]

In [3]:

import pandas as pd
from pathlib import Path
import geopandas as gpd

#region_list = ["RegionA", "RegionB", "RegionC"]  # your region names

results = []  # store region + total flooded length

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Damages_Artefact_Bridges_Viaducts_Tunnels_removed.gpkg"

    # Load roads layer
    gdf = gpd.read_file(roads_ex)

    # Compute flooded length
    gdf["flooded_length"] = gdf["F_EV1_fr"] * gdf["length"]

    # Sum for region
    total_flooded = gdf["flooded_length"].sum()

    # Store
    results.append({
        "Region": region,
        "Flooded Length Road": total_flooded
    })

# Convert results to table
df_results = pd.DataFrame(results)

print(df_results)




Processing region: ARK-NZK network
Processing region: Vallei en Veluwe network
Processing region: Achterhoek network
Processing region: Brabantse Delta network
Processing region: Friesland network
Processing region: Groningen en NO-Drenthe network
Processing region: Limburg network
Processing region: Noord-Brabant Oost network
Processing region: Noord-Westelijke Delta network
Processing region: Rivierenland network
Processing region: Scheldestromen network
Processing region: Zuiderzeeland network
Processing region: Overijsselse Vecht network
                     Region  Flooded Length Road
0                   ARK-NZK         11210.913886
1          Vallei en Veluwe        102387.881451
2                Achterhoek         10829.800036
3           Brabantse Delta         22394.384254
4                 Friesland         11927.920300
5   Groningen en NO-Drenthe         27571.200815
6                   Limburg         57927.943292
7        Noord-Brabant Oost         54127.188867
8    Noord-

In [ ]:
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Aggregated_schakels.gpkg"

In [ ]:

import geopandas as gpd
from pathlib import Path
import pandas as pd

# Input files
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")

results = []

for region in region_list:
    print(f"Processing region: {region} network")

    # ---------------------------
    # 1. Load polygon for region
    # ---------------------------
    areas = gpd.read_file(area_gpkg)
    region_poly = areas[areas["name"] == region]

    if region_poly.empty:
        print(f"⚠️ Region polygon not found for: {region}")
        continue

    # ---------------------------
    # 2. Load aggregated network
    # ---------------------------
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Aggregated_schakels.gpkg"

    gdf = gpd.read_file(roads_ex)

    # Ensure same CRS
    if gdf.crs != region_poly.crs:
        region_poly = region_poly.to_crs(gdf.crs)

    # ---------------------------
    # 3. Clip lines to polygon
    # ---------------------------
    clipped = gpd.clip(gdf, region_poly)

    # ---------------------------
    # 4. Calculate length
    # ---------------------------
    clipped["length_m"] = clipped.geometry.length

    total_length = clipped["length_m"].sum()

    # Store in results table
    results.append({
        "Region": region,
        "Total Length (m)": total_length
    })

# ---------------------------
# 5. Convert to summary table
# ---------------------------
summary_df = pd.DataFrame(results)

print("\n=== Summary Table ===")
print(summary_df)


# Optional: nicer formatting (keeps numeric type, only affects display if you round here)
summary_df["Total Length (m)"] = summary_df["Total Length (m)"].round(2)

# 6) Save to Excel
out_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\_summary")
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx = out_dir / "regional_total_length.xlsx"
summary_df.to_excel(out_xlsx, index=False)

print(f"\n✅ Saved summary to: {out_xlsx}")


Processing region: ARK-NZK network
Processing region: Vallei en Veluwe network
Processing region: Achterhoek network
Processing region: Brabantse Delta network
Processing region: Friesland network
Processing region: Groningen en NO-Drenthe network
Processing region: Limburg network
Processing region: Noord-Brabant Oost network
Processing region: Noord-Westelijke Delta network
Processing region: Rivierenland network
Processing region: Scheldestromen network
Processing region: Zuiderzeeland network
Processing region: Overijsselse Vecht network

=== Summary Table ===
                     Region  Total Length (m)
0                   ARK-NZK      1.497132e+06
1          Vallei en Veluwe      7.085102e+05
2                Achterhoek      2.891590e+05
3           Brabantse Delta      5.833083e+05
4                 Friesland      5.322479e+05
5   Groningen en NO-Drenthe      4.621565e+05
6                   Limburg      5.788211e+05
7        Noord-Brabant Oost      7.656226e+05
8    Noord-West